In [ ]:
import os
import json
import random

from datasets import load_from_disk, load_dataset
from datasets.features import Value, ClassLabel

from utils import dataset_class_encode_column, validate_equal_datasets, extend_datasets

# Dataset for image classification

In [ ]:
datasets2concat = {
    "milk10k": "/home/sulcm/datasets/milk10k/milk10k",
    "isic2019": "/home/sulcm/datasets/isic2019/isic2019",
}
valid_columns = [
    "label",
    "image",
]

im_cls_metadata = {}
with open("../training/image_classification/configs/metadata.json", "r") as f:
    _ld_metadata = json.load(f)
    if isinstance(_ld_metadata, dict):
        im_cls_metadata.update(**_ld_metadata)
    else:
        im_cls_metadata["additional_metadata"] = _ld_metadata

In [ ]:
ld_datasets = []
label2id = {l: i for i, l in enumerate(im_cls_metadata["labels"])}
for d_name, d_path in datasets2concat.items():
    if os.path.exists(d_path):
        _ds = load_from_disk(d_path)
    else:
        _ds = load_dataset(d_path)

    for split in _ds.keys():
        _ds[split] = _ds[split].remove_columns(
            column_names=[c_name for c_name in _ds[split].column_names if c_name not in valid_columns]
        )
        _ds[split] = _ds[split].add_column(name="original_ds_row_id", column=list(range(len(_ds[split]))))
        _ds[split] = _ds[split].add_column(name="original_ds_name", column=[d_name]*len(_ds[split]))

    if all([isinstance(s.features["label"], ClassLabel) for s in _ds.values()]):
        new_ds = _ds.align_labels_with_mapping(label2id, "label")
    else:
        new_ds = dataset_class_encode_column(
            dataset=_ds,
            column="label",
            custom_labels=im_cls_metadata["labels"]
        )

    res, msg = validate_equal_datasets(_ds, new_ds, columns=valid_columns)
    if res:
        print(f"Dataset {d_name} was loaded, processed, and successfully validated. Adding to list ...")
    else:
        raise Value(f"Dataset {d_name} is not valid: {msg}")

    ld_datasets.append(new_ds)

In [ ]:
dset_concat = extend_datasets(ld_datasets)

In [ ]:
# dset_concat.save_to_disk("/home/sulcm/datasets/MelanoMix")

# Balance class distribution between train and validation sets
- supply percentage of missing labels in validation portion of datasets from training dataset
- the samples are selected randomly

In [ ]:
melano_mix = load_from_disk("/home/sulcm/datasets/MelanoMix")
melano_mix

In [ ]:
labels = melano_mix["train"].features["label"].names
set_labels = set(labels)
assert len(set_labels) == len(labels), "Repeating class labels"
assert set_labels == set(melano_mix["validation"].features["label"].names), "Missmatch between train and validation labels"

In [ ]:
train_labels = set(melano_mix["train"]["label"])
val_labels = set(melano_mix["validation"]["label"])

missing_labels = train_labels - val_labels
missing_labels

In [ ]:
def select_sample_by_label(sample):
    return sample["label"] == 2

In [ ]:
missing_samples_idx = melano_mix["train"].filter(select_sample_by_label)

In [ ]:
label_indices = missing_samples_idx._indices.to_pandas()["indices"].tolist()

In [ ]:
n_to_add = max(1, int(len(label_indices) * 0.1))  # e.g., 10% of samples
sampled_indices = random.sample(label_indices, n_to_add)

In [ ]:
n_to_add

In [ ]:
sampled_indices